In [1]:
from pytential.sympy_pytential import sympy_pytential
import numpy as np
import matplotlib.pyplot as plt
from pytential.reduce.matrix_methods import reduce_qp

Create two ideal mixing functions from a set of properties, and check them.

In [2]:
from sympy import log, symbols
c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd = symbols('c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd')

In [3]:
T = 1600
RT = 8.134*T
mu0_SiC = -161028
rho_SiC = 3.21 / 40.11 * 1e6
rho_Ar = 101e3 / RT
v_SiC = 1/rho_SiC
v_Ar = 1/rho_Ar

In [4]:
fa_sp = c0a*mu0_SiC 
fb_sp = c0b*mu0_SiC
fc_sp = c0c*mu0_SiC 
fd_sp = c0d*(-120100+RT*log(c0d/(c0d+c1d))) +c1d*(-290457+RT*log(c1d/(c0d+c1d)))

In [5]:
np.exp((mu0_SiC--120100)/RT)

0.04307449610427493

In [6]:
fa = sympy_pytential(fa_sp, constraints_sym=[c0a*v_SiC-Va])
fb = sympy_pytential(fb_sp, constraints_sym=[c0b*v_SiC-Vb])
fc = sympy_pytential(fc_sp, constraints_sym=[c0c*v_SiC-Vc])
fd = sympy_pytential(fd_sp, constraints_sym=[c0d*v_Ar+c1d*v_Ar-Vd])

Make a function fa+fb with all the variables, and add constraints that the concentrations must sum to ca and cb

In [7]:
f = fa+fb+fc+fd
f = f.add_constraints_sym([c0a+c0b+c0c+c0d-c0, c1d-c1])
print(f)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + c0d*(13014.4*log(c0d/(c0d + c1d)) - 120100) + c1d*(13014.4*log(c1d/(c0d + c1d)) - 290457)

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 13014.4*log(c0d/(c0d + c1d)) - 120100.0, 0, 13014.4*log(c1d/(c0d + c1d)) - 290457.0]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 13014.4*c1d/(c0d*(c0d + c1d)), 0, -13014.4/(c0d + c1d)], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -13014.4/(c0d + c1d), 0, 13014.4*c0d/(c1d*(c0d + c1d))]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1.24953271028037e-5*c0b
-Vc + 1.24953271028037e-5*c0c
-Vd + 0.128855445544554*c0d + 0.12885544554455

In [8]:
1.24953271028037e-5*140333

1.7535067383177516

In [9]:
y0 = {'Va':1, 'Vb':1, 'Vc':1, 'Vd':1, 'c0a':rho_SiC, 'c0b':rho_SiC, 'c0c':rho_SiC , 'c0d':rho_Ar*0.043, 'c1d':rho_Ar*(1-0.043), 'c0':0,'c1':0}
print(y0)

{'Va': 1, 'Vb': 1, 'Vc': 1, 'Vd': 1, 'c0a': 80029.9177262528, 'c0b': 80029.9177262528, 'c0c': 80029.9177262528, 'c0d': 0.3337072780919596, 'c1d': 7.42692709613966, 'c0': 0, 'c1': 0}


In [10]:
fq = f.quadratic_expansion(y0)
print(fq)
print(fq.vars)
print(fq.grad(**y0))



Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + 0.5*c0d*(37322.4727707852*c0d - 1676.97631049505*c1d) - 161050.527517103*c0d + 0.5*c1d*(-1676.97631049505*c0d + 75.3500327599657*c1d) - 291029.00744506*c1d - 38663387969.8234

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 37322.4727707852*c0d - 1676.97631049505*c1d - 161050.527517103, 0, -1676.97631049505*c0d + 75.3500327599657*c1d - 291029.00744506]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 37322.4727707852, 0, -1676.97631049505], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -1676.97631049505, 0, 75.3500327599657]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1

In [11]:
fr = fq.remove_linear_constraints(['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd'], y0=y0)
print(fr)
print(fr.vars)
print(fr.grad(**y0))
y1 = {'Va': 1, 'Vb': 0, 'Vc': 0, 'Vd': 0, 'c0': 180029.9177262528, 'c1': 0}
print(fr(**y1))
print(fr.grad(**y1))

Condition number of A_d:  160143.58675631855
Q_tilde is symmetric.

Variables
['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd']

Potential
0.5*Va*(-0.00243424853454655*Va + 73.9194436731646*Vb + 3.03738754980909e-8*Vc - 9.95289744014346*Vd - 0.00243269948324343*c0 - 0.00242550822191239*c1) + 1.76204248262613*Va + 0.5*Vb*(-0.00243424916943643*Va + 73.9194629525053*Vb + 3.03738839357859e-8*Vc - 9.95290003601393*Vd - 0.00243270011772929*c0 - 0.00242550885452266*c1) + 174.828005620278*Vb + 0.5*Vc*(-0.00243424875485628*Va + 73.9194503631851*Vb + 3.03738778573148e-8*Vc - 9.95289834092254*Vd - 0.00243269970341296*c0 - 0.00242550844143108*c1) + 161027.999976545*Vc + 0.5*Vd*(74.023749717857*Va - 2247837.19496614*Vb - 0.000923647739000444*Vc + 302660.463227483*Vd + 73.9766441802199*c0 + 73.7579630877307*c1) + 291006.479904502*Vd + 0.5*c0*(3.04167333331736e-8*Va - 0.000923647677890879*Vb - 3.79531531172244e-13*Vc + 0.000124364715859083*Vd + 3.03973774293957e-8*c0 + 3.03075202619244e-8*c1) + 1.76813741703148*c